# Notebook 10: Research Validation and Final Checks

## Objective
Validate the existing Caco-2 chemical-space research results, resolve pending methodological questions, and document final decisions before manuscript preparation.

## Research question
Does molecular representation influence the relationship between chemical-space distance and prediction error in Caco-2 permeability prediction?

## Validation principles
- Preserve all original notebooks, datasets, results, and figures.
- Do not retrain models or repeat completed analyses without a specific justification.
- Verify saved results against their underlying calculations wherever possible.
- Distinguish confirmed results, unresolved questions, and necessary corrections.
- Record all validation decisions and their scientific rationale.

## Planned validation
1. Inventory existing research files and saved outputs.
2. Verify molecular representations and distance definitions.
3. Validate nearest-neighbor distance calculations.
4. Check random-split correlations and Q1–Q5 results.
5. Check scaffold-split correlations and Q1–Q5 results.
6. Review statistical tests, multiple-testing corrections, and bootstrap comparisons.
7. Decide whether training-set-distance analysis is necessary.
8. Produce a final validation report and manuscript-readiness checklist.

## Important restriction
Existing results will not be overwritten. Any required corrections will be saved separately and documented.

# Inventory the existing files

In [2]:
from pathlib import Path
import pandas as pd

# Research folder
PAPER_DIR = Path(
    r"C:\Users\kiran\Cheminformatics_course\Paper"
)

print("Folder exists:", PAPER_DIR.exists())
print("\nResearch files:\n")

for path in sorted(PAPER_DIR.iterdir()):
    if path.is_file():
        print(f"{path.name}  |  {path.stat().st_size:,} bytes")

Folder exists: True

Research files:

02_Caco2_Papp_Curated.csv  |  4,097,205 bytes
02_Data_Set_Audit.ipynb  |  269,339 bytes
03_EDA.ipynb  |  419,060 bytes
04_TDA.ipynb  |  998,273 bytes
05_Chemical_Space_Representation.ipynb  |  168,564 bytes
06_Baseline Prediction Model.ipynb  |  183,293 bytes
07_XGBoost_Representation_Comparison.ipynb  |  117,591 bytes
08_Scaffold-Based_Generalization_Analysis,.ipynb  |  93,630 bytes
09_Final Results Consolidation and Publication-Quality Figures.ipynb  |  659,054 bytes
Caco2_chemical_space_master.pkl  |  1,961,598 bytes
Caco2_chemical_space_primary_results.pkl  |  1,322 bytes
caco2_morgan_chemical_space.pkl  |  60,780,403 bytes
Caco2_Papp_20449_single_fragment_dataset.csv  |  7,161,824 bytes
chembl_hia_raw_from_api.csv  |  4,779,191 bytes
consolidated_distance_error_correlations.csv  |  327 bytes
consolidated_distance_quintile_endpoints.csv  |  385 bytes
consolidated_model_performance.csv  |  537 bytes
Figure1_model_performance.png  |  328,564 byte

In [3]:
import json
import pandas as pd

# Statistical validation summary
with open("final_statistical_validation.json", "r") as f:
    statistical_results = json.load(f)

print("=== Statistical Validation JSON ===")
print(json.dumps(statistical_results, indent=2))

print("\n=== Consolidated Correlations ===")
print(pd.read_csv("consolidated_distance_error_correlations.csv").to_string(index=False))

print("\n=== Quantile Endpoints ===")
print(pd.read_csv("consolidated_distance_quintile_endpoints.csv").to_string(index=False))

print("\n=== Model Performance ===")
print(pd.read_csv("consolidated_model_performance.csv").to_string(index=False))

=== Statistical Validation JSON ===
{
  "n_molecules": 20449,
  "n_test_molecules": 4090,
  "friedman_test": {
    "statistic": 30.5485,
    "p": 2.325279e-07
  },
  "pairwise_wilcoxon": {
    "Morgan_vs_RDKit": {
      "raw_p": 4e-06,
      "Holm_p": 1.1e-05,
      "effect": -0.083519
    },
    "Morgan_vs_Hybrid": {
      "raw_p": 0.000573,
      "Holm_p": 0.001146,
      "effect": -0.06218
    },
    "RDKit_vs_Hybrid": {
      "raw_p": 0.002946,
      "Holm_p": 0.002946,
      "effect": 0.053702
    }
  },
  "standardized_slopes": {
    "Morgan": 0.06309,
    "RDKit": 0.085409,
    "Hybrid": 0.069732
  },
  "bootstrap_slope_comparison": {
    "RDKit_vs_Morgan": {
      "difference": 0.024562,
      "CI95": [
        0.002388,
        0.048211
      ]
    },
    "RDKit_vs_Hybrid": {
      "difference": 0.017904,
      "CI95": [
        -0.00345,
        0.039174
      ]
    },
    "Hybrid_vs_Morgan": {
      "difference": 0.006657,
      "CI95": [
        -0.002334,
        0.015674


# Inspect the saved scaffold quantile results

In [4]:
scaffold_quantiles = pd.read_csv(
    "scaffold_distance_quintile_analysis.csv"
)

print("Shape:", scaffold_quantiles.shape)
print("Columns:", scaffold_quantiles.columns.tolist())
print(scaffold_quantiles.to_string(index=False))

Shape: (3, 8)
Columns: ['Representation', 'Q1_mean_error', 'Q5_mean_error', 'Increase_percent', 'U_statistic', 'p_value', 'p_holm', 'significant_after_holm']
Representation  Q1_mean_error  Q5_mean_error  Increase_percent  U_statistic      p_value       p_holm  significant_after_holm
        Morgan       0.425239       0.503847         18.485658     307174.0 1.683328e-07 3.366656e-07                    True
         RDKit       0.390203       0.533349         36.684887     289124.0 1.025603e-12 3.076809e-12                    True
        Hybrid       0.426442       0.509119         19.387548     309103.0 2.566026e-07 3.366656e-07                    True


# inspect the random-split quintile statistics

# Load the saved results

In [6]:
import pickle

with open("Caco2_chemical_space_primary_results.pkl", "rb") as f:
    primary_results = pickle.load(f)

print("Primary results loaded successfully.")
print("Available sections:", list(primary_results.keys()))

Primary results loaded successfully.
Available sections: ['n_molecules', 'n_test_molecules', 'Morgan_features', 'RDKit_features', 'Hybrid_features', 'model_performance', 'distance_error_spearman', 'q1_q5', 'friedman_test', 'pairwise_wilcoxon', 'standardized_slopes', 'bootstrap_slope_comparison', 'outlier_robustness']


# Inspect the random-split Q1–Q5 results

In [7]:
for rep, values in primary_results["q1_q5"].items():
    print(f"\n{rep}")
    for key, value in values.items():
        print(f"  {key}: {value}")


Morgan
  Q1_mean_error: 0.3249
  Q5_mean_error: 0.4878
  increase_percent: 50.1
  rank_biserial: 0.2876
  p: 6.271338e-24

RDKit
  Q1_mean_error: 0.2964
  Q5_mean_error: 0.5351
  increase_percent: 80.6
  rank_biserial: 0.3549
  p: 1.854186e-35

Hybrid
  Q1_mean_error: 0.318
  Q5_mean_error: 0.5014
  increase_percent: 57.7
  rank_biserial: 0.274
  p: 8.430179e-22


# Find the original random-split statistical results

In [11]:
print("Sections in statistical validation:")
print(list(statistical_results.keys()))

print("\nDoes it contain Q1–Q5 results?")
print("q1_q5" in statistical_results)

Sections in statistical validation:
['n_molecules', 'n_test_molecules', 'friedman_test', 'pairwise_wilcoxon', 'standardized_slopes', 'bootstrap_slope_comparison', 'outlier_robustness']

Does it contain Q1–Q5 results?
False


In [10]:
import json

with open("final_statistical_validation.json", "r") as f:
    statistical_results = json.load(f)

## Validation Finding: Random-Split Q1–Q5 Analysis

The representation-specific quintile summaries and Mann–Whitney U test outputs were located in Notebook 06. The Q1–Q5 mean absolute errors match the values stored in the primary results pickle.

| Representation | Q1 mean error | Q5 mean error | Raw p-value | Rank-biserial effect |
|---|---:|---:|---:|---:|
| Morgan | 0.3249 | 0.4878 | 6.271338e-24 | 0.2876 |
| RDKit | 0.2964 | 0.5351 | 1.854186e-35 | 0.3549 |
| Hybrid | 0.3180 | 0.5014 | 8.430179e-22 | 0.2740 |

All three comparisons show substantially higher prediction error in Q5 than Q1. RDKit has the largest observed Q1–Q5 error increase and rank-biserial effect.

Pending verification: The displayed outputs contain raw p-values. Holm-adjusted p-values for these random-split comparisons have not yet been confirmed.

No model retraining or quintile recalculation was performed during this validation step.

## Validation Findings: Pairwise Effects, Robustness, and Bootstrap Comparisons

### Pairwise model-error comparisons
The saved Wilcoxon comparisons are statistically significant after Holm correction, but the corresponding rank-biserial effect sizes are small in magnitude.

### Outlier robustness
After excluding the top 1% of molecules by distance, the Spearman correlation ranking remains RDKit > Hybrid > Morgan. The correlations change only modestly, supporting robustness to this specific outlier exclusion procedure.

### Bootstrap slope comparisons
The 95% bootstrap confidence interval for RDKit minus Morgan excludes zero. The intervals for RDKit minus Hybrid and Hybrid minus Morgan include zero. Therefore, only the RDKit–Morgan slope difference is supported by these reported intervals.

### Remaining statistical question
Holm-adjusted p-values for the three random-split Q1-versus-Q5 Mann–Whitney tests have not yet been confirmed.

In [1]:
import pandas as pd
import numpy as np
import pickle

from sklearn.model_selection import GroupShuffleSplit

print("Libraries loaded successfully.")

Libraries loaded successfully.


In [2]:
import pickle

with open("Caco2_chemical_space_master.pkl", "rb") as f:
    master_data = pickle.load(f)

print("Object type:", type(master_data))

if isinstance(master_data, dict):
    print("Keys:", master_data.keys())

Object type: <class 'pandas.core.frame.DataFrame'>


# Inspect its columns

In [3]:
df = master_data.copy()

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

Dataset shape: (20449, 5)

Columns:
['canonical_smiles', 'log10_median_papp', 'Morgan_distance', 'RDKit_distance', 'Hybrid_distance']


# Load the curated dataset

In [4]:
curated_df = pd.read_csv("02_Caco2_Papp_Curated.csv")

print("Dataset shape:", curated_df.shape)
print("\nColumns:")
print(curated_df.columns.tolist())

Dataset shape: (29151, 10)

Columns:
['assay_chembl_id', 'canonical_smiles', 'molecule_chembl_id', 'standard_type', 'standard_units', 'standard_value', 'type', 'units', 'value', 'papp_u6_cm_s']


In [5]:
df = pd.read_csv(
    "Caco2_Papp_20449_single_fragment_dataset.csv"
)

print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())
print("Scaffold column exists:", "scaffold" in df.columns)

Dataset shape: (20449, 15)
Columns: ['canonical_smiles', 'n_measurements', 'n_assays', 'min_papp', 'max_papp', 'median_papp', 'mean_papp', 'log10_range', 'log10_median_papp', 'mean_median_ratio', 'mol', 'rdkit_canonical_smiles', 'n_fragments', 'parent_mol', 'parent_smiles']
Scaffold column exists: False


# Recreate the scaffolds

In [8]:
from rdkit import Chem
from rdkit.Chem.Scaffolds import MurckoScaffold

# Extract scaffolds exactly as in Notebook 08
def get_scaffold(smiles):
    mol = Chem.MolFromSmiles(smiles)

    if mol is None:
        return None

    scaffold = MurckoScaffold.GetScaffoldForMol(mol)

    return Chem.MolToSmiles(
        scaffold,
        canonical=True
    )

df["scaffold"] = df["canonical_smiles"].apply(get_scaffold)

print("Total molecules:", len(df))
print("Invalid/missing scaffolds:", df["scaffold"].isna().sum())
print("Unique scaffolds:", df["scaffold"].nunique())

Total molecules: 20449
Invalid/missing scaffolds: 0
Unique scaffolds: 10698


# Recreate the original scaffold split

In [9]:
from sklearn.model_selection import GroupShuffleSplit

y_scaffold = df["log10_median_papp"].values
groups = df["scaffold"].values

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx_scaffold, test_idx_scaffold = next(
    splitter.split(df, y_scaffold, groups=groups)
)

train_scaffolds = set(df.iloc[train_idx_scaffold]["scaffold"])
test_scaffolds = set(df.iloc[test_idx_scaffold]["scaffold"])

print("Training molecules:", len(train_idx_scaffold))
print("Test molecules:", len(test_idx_scaffold))
print("Overlapping scaffolds:", len(train_scaffolds & test_scaffolds))

Training molecules: 16202
Test molecules: 4247
Overlapping scaffolds: 0


In [10]:
pred = pd.read_csv("scaffold_split_predictions.csv")

print("Prediction rows:", len(pred))
print("Columns:")
print(pred.columns.tolist())
print("\nFirst 3 rows:")
display(pred.head(3))

Prediction rows: 4247
Columns:
['canonical_smiles', 'scaffold', 'log10_median_papp', 'y_true', 'Morgan_pred', 'Morgan_error', 'RDKit_pred', 'RDKit_error', 'Hybrid_pred', 'Hybrid_error']

First 3 rows:


,canonical_smiles,scaffold,log10_median_papp,y_true,Morgan_pred,Morgan_error,RDKit_pred,RDKit_error,Hybrid_pred,Hybrid_error
0,Brc1ccc(-c2nc3ccc(Br)cn3n2)cc1,c1ccc(-c2nc3ccccn3n2)cc1,0.542825,0.542825,1.208765,0.665940,1.187154,0.644328,1.201493,0.658668
1,Brc1ccc(C2(CC3CCOCC3)c3ccccc3-c3nccn32)cc1,c1ccc(C2(CC3CCOCC3)c3ccccc3-c3nccn32)cc1,1.491362,1.491362,1.276727,0.214634,1.249238,0.242124,1.181594,0.309768
2,Brc1ccc2[nH]nc(C3=CCCNC3)c2c1,C1=C(c2n[nH]c3ccccc23)CNCC1,0.897627,0.897627,1.007112,0.109485,0.995471,0.097844,1.011336,0.113709


# Verify molecule alignment

In [11]:
# Compare test molecules with the saved prediction molecules
test_smiles = set(df.iloc[test_idx_scaffold]["canonical_smiles"])
pred_smiles = set(pred["canonical_smiles"])

print("Test molecules:", len(test_smiles))
print("Saved prediction molecules:", len(pred_smiles))
print("Molecules missing from predictions:", len(test_smiles - pred_smiles))
print("Extra molecules in predictions:", len(pred_smiles - test_smiles))
print("Duplicate prediction SMILES:", pred["canonical_smiles"].duplicated().sum())

Test molecules: 4247
Saved prediction molecules: 4247
Molecules missing from predictions: 0
Extra molecules in predictions: 0
Duplicate prediction SMILES: 0


# Load the existing molecular representations

In [12]:
with open("caco2_morgan_chemical_space.pkl", "rb") as f:
    morgan_data = pickle.load(f)

print("Object type:", type(morgan_data))

if isinstance(morgan_data, dict):
    print("Keys:", morgan_data.keys())

Object type: <class 'dict'>
Keys: dict_keys(['df_cs', 'X_morgan', 'nearest_indices', 'nearest_similarities', 'nearest_distances'])


# Check the fingerprint matrix

In [13]:
X_morgan = morgan_data["X_morgan"]

print("Morgan matrix shape:", X_morgan.shape)
print("Matrix type:", type(X_morgan))
print("Dataset rows:", len(df))

Morgan matrix shape: (20449, 2048)
Matrix type: <class 'numpy.ndarray'>
Dataset rows: 20449


# Verify Morgan row alignment

In [14]:
morgan_df = morgan_data["df_cs"]

print("Morgan dataframe shape:", morgan_df.shape)
print("Morgan dataframe columns:", morgan_df.columns.tolist())
print("\nFirst 3 Morgan SMILES:")
print(morgan_df["canonical_smiles"].head(3).tolist())

print("\nFirst 3 dataset SMILES:")
print(df["canonical_smiles"].head(3).tolist())

Morgan dataframe shape: (20449, 16)
Morgan dataframe columns: ['canonical_smiles', 'n_measurements', 'n_assays', 'min_papp', 'max_papp', 'median_papp', 'mean_papp', 'log10_range', 'log10_median_papp', 'mean_median_ratio', 'mol', 'rdkit_canonical_smiles', 'n_fragments', 'parent_mol', 'parent_smiles', 'morgan_fp']

First 3 Morgan SMILES:
['Brc1ccc(-c2nc(-c3ccc4nc[nH]c4c3)no2)cc1', 'Brc1ccc(-c2nc3ccc(Br)cn3n2)cc1', 'Brc1ccc(-c2nnc(N3CCN(c4ccccn4)CC3)o2)cc1']

First 3 dataset SMILES:
['Brc1ccc(-c2nc(-c3ccc4nc[nH]c4c3)no2)cc1', 'Brc1ccc(-c2nc3ccc(Br)cn3n2)cc1', 'Brc1ccc(-c2nnc(N3CCN(c4ccccn4)CC3)o2)cc1']


# Full Morgan alignment check

In [15]:
morgan_smiles = morgan_df["canonical_smiles"].reset_index(drop=True)
dataset_smiles = df["canonical_smiles"].reset_index(drop=True)

print("Full row order matches:", morgan_smiles.equals(dataset_smiles))

Full row order matches: True


# Calculate Morgan training-only distances

In [16]:
from rdkit import DataStructs
from tqdm.auto import tqdm

# Convert stored fingerprints to RDKit bit vectors if needed
fps = morgan_df["morgan_fp"].tolist()

train_fps = [fps[i] for i in train_idx_scaffold]
test_fps = [fps[i] for i in test_idx_scaffold]

morgan_train_distances = []

for fp in tqdm(test_fps, desc="Morgan training-only distances"):
    similarities = DataStructs.BulkTanimotoSimilarity(fp, train_fps)
    morgan_train_distances.append(1 - max(similarities))

pred["Morgan_train_distance"] = morgan_train_distances

print("Test distances:", len(morgan_train_distances))
print("Missing distances:", pred["Morgan_train_distance"].isna().sum())
print(pred["Morgan_train_distance"].describe())

C:\Users\kiran\anaconda3\envs\chemtox_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Morgan training-only distances: 100%|██████████| 4247/4247 [00:18<00:00, 235.93it/s]

Test distances: 4247
Missing distances: 0
count    4247.000000
mean        0.353206
std         0.178117
min         0.000000
25%         0.226541
50%         0.324324
75%         0.471346
max         0.833333
Name: Morgan_train_distance, dtype: float64


# Calculate RDKit training-only distances

In [17]:
with open("RDKit_descriptor_chemical_space.pkl", "rb") as f:
    rdkit_data = pickle.load(f)

print("Keys:", rdkit_data.keys())
print("Scaled descriptors shape:",
      rdkit_data["descriptors_scaled"].shape)

Keys: dict_keys(['molecules', 'descriptors', 'descriptors_scaled', 'descriptor_names', 'nearest_neighbor_distance'])
Scaled descriptors shape: (20449, 205)


# Verify RDKit row alignment

In [18]:
rdkit_molecules = rdkit_data["molecules"]

print("RDKit molecule count:", len(rdkit_molecules))
print("First 3 RDKit molecules:")
print(rdkit_molecules[:3])

print("\nFirst 3 dataset SMILES:")
print(df["canonical_smiles"].head(3).tolist())

RDKit molecule count: 20449
First 3 RDKit molecules:
['Brc1ccc(-c2nc(-c3ccc4nc[nH]c4c3)no2)cc1'
 'Brc1ccc(-c2nc3ccc(Br)cn3n2)cc1'
 'Brc1ccc(-c2nnc(N3CCN(c4ccccn4)CC3)o2)cc1']

First 3 dataset SMILES:
['Brc1ccc(-c2nc(-c3ccc4nc[nH]c4c3)no2)cc1', 'Brc1ccc(-c2nc3ccc(Br)cn3n2)cc1', 'Brc1ccc(-c2nnc(N3CCN(c4ccccn4)CC3)o2)cc1']


In [19]:
rdkit_smiles = pd.Series(
    rdkit_data["molecules"]
).reset_index(drop=True)

dataset_smiles = df["canonical_smiles"].reset_index(drop=True)

print("Full row order matches:", rdkit_smiles.equals(dataset_smiles))

Full row order matches: True


# Calculate RDKit training-only distances

In [20]:
from sklearn.metrics import pairwise_distances
from tqdm.auto import tqdm

X_rdkit = rdkit_data["descriptors_scaled"]

X_train_rdkit = X_rdkit[train_idx_scaffold]
X_test_rdkit = X_rdkit[test_idx_scaffold]

# Minimum Euclidean distance to any training molecule
rdkit_train_distances = []

for start in tqdm(range(0, len(X_test_rdkit), 256)):
    batch = X_test_rdkit[start:start + 256]

    distances = pairwise_distances(
        batch,
        X_train_rdkit,
        metric="euclidean"
    )

    rdkit_train_distances.extend(distances.min(axis=1))

pred["RDKit_train_distance"] = rdkit_train_distances

print("Test distances:", len(rdkit_train_distances))
print("Missing distances:", pred["RDKit_train_distance"].isna().sum())
print(pred["RDKit_train_distance"].describe())

100%|██████████| 17/17 [00:01<00:00, 10.40it/s]

Test distances: 4247
Missing distances: 0
count    4247.000000
mean        5.451408
std         3.087029
min         0.000000
25%         3.720296
50%         5.243604
75%         6.918448
max        71.684724
Name: RDKit_train_distance, dtype: float64


# Load the Hybrid representation

In [21]:
with open("Hybrid_chemical_space.pkl", "rb") as f:
    hybrid_data = pickle.load(f)

print("Keys:", hybrid_data.keys())
print(
    "Hybrid representation shape:",
    hybrid_data["hybrid_representation"].shape
)

Keys: dict_keys(['molecules', 'morgan_normalized', 'rdkit_normalized', 'hybrid_representation', 'nearest_neighbor_distance', 'nearest_neighbor_index'])
Hybrid representation shape: (20449, 2253)


# Verify Hybrid row alignment

In [22]:
hybrid_smiles = pd.Series(
    hybrid_data["molecules"]
).reset_index(drop=True)

dataset_smiles = df["canonical_smiles"].reset_index(drop=True)

print("Hybrid molecule count:", len(hybrid_smiles))
print("Full row order matches:", hybrid_smiles.equals(dataset_smiles))

Hybrid molecule count: 20449
Full row order matches: True


# Calculate Hybrid training-only distances

In [23]:
from sklearn.metrics import pairwise_distances
from tqdm.auto import tqdm

X_hybrid = hybrid_data["hybrid_representation"]

X_train_hybrid = X_hybrid[train_idx_scaffold]
X_test_hybrid = X_hybrid[test_idx_scaffold]

hybrid_train_distances = []

for start in tqdm(range(0, len(X_test_hybrid), 256)):
    batch = X_test_hybrid[start:start + 256]

    distances = pairwise_distances(
        batch,
        X_train_hybrid,
        metric="euclidean"
    )

    hybrid_train_distances.extend(distances.min(axis=1))

pred["Hybrid_train_distance"] = hybrid_train_distances

print("Test distances:", len(hybrid_train_distances))
print("Missing distances:", pred["Hybrid_train_distance"].isna().sum())
print(pred["Hybrid_train_distance"].describe())

100%|██████████| 17/17 [00:07<00:00,  2.26it/s]

Test distances: 4247
Missing distances: 0
count    4247.000000
mean        0.839327
std         0.298377
min         0.000000
25%         0.635864
50%         0.824204
75%         1.045910
max         1.491658
Name: Hybrid_train_distance, dtype: float64


# Verify the three distance columns

In [24]:
distance_cols = [
    "Morgan_train_distance",
    "RDKit_train_distance",
    "Hybrid_train_distance"
]

print("Rows:", len(pred))
print("\nMissing values:")
print(pred[distance_cols].isna().sum())

print("\nDistance summary:")
display(pred[distance_cols].describe().round(4))

Rows: 4247

Missing values:
Morgan_train_distance    0
RDKit_train_distance     0
Hybrid_train_distance    0
dtype: int64

Distance summary:


,Morgan_train_distance,RDKit_train_distance,Hybrid_train_distance
count,4247.0000,4247.0000,4247.0000
mean,0.3532,5.4514,0.8393
std,0.1781,3.0870,0.2984
min,0.0000,0.0000,0.0000
25%,0.2265,3.7203,0.6359
50%,0.3243,5.2436,0.8242
75%,0.4713,6.9184,1.0459
max,0.8333,71.6847,1.4917


# Analyze distance versus prediction error

In [25]:
from scipy.stats import spearmanr

results = []

for rep in ["Morgan", "RDKit", "Hybrid"]:
    distance = pred[f"{rep}_train_distance"]
    error = pred[f"{rep}_error"]

    rho, p_value = spearmanr(distance, error)

    results.append({
        "Representation": rep,
        "Spearman_rho": rho,
        "p_value": p_value
    })

distance_error_results = pd.DataFrame(results)

display(distance_error_results.round(8))

,Representation,Spearman_rho,p_value
0,Morgan,0.179306,0.0
1,RDKit,0.211808,0.0
2,Hybrid,0.146648,0.0


# Compare Q1 versus Q5

In [26]:
from scipy.stats import mannwhitneyu

q1q5_train_results = []

for rep in ["Morgan", "RDKit", "Hybrid"]:
    distance_col = f"{rep}_train_distance"
    error_col = f"{rep}_error"

    temp = pred[[distance_col, error_col]].copy()
    temp["quintile"] = pd.qcut(
        temp[distance_col],
        q=5,
        labels=["Q1", "Q2", "Q3", "Q4", "Q5"],
        duplicates="drop"
    )

    q1 = temp.loc[temp["quintile"] == "Q1", error_col]
    q5 = temp.loc[temp["quintile"] == "Q5", error_col]

    u_stat, p_value = mannwhitneyu(
        q1, q5, alternative="two-sided"
    )

    q1q5_train_results.append({
        "Representation": rep,
        "Q1_mean_error": q1.mean(),
        "Q5_mean_error": q5.mean(),
        "Increase_percent": (q5.mean() - q1.mean()) / q1.mean() * 100,
        "U_statistic": u_stat,
        "p_value": p_value
    })

q1q5_train = pd.DataFrame(q1q5_train_results)
display(q1q5_train.round(6))

,Representation,Q1_mean_error,Q5_mean_error,Increase_percent,U_statistic,p_value
0,Morgan,0.366646,0.557186,51.968178,262164.0,0.0
1,RDKit,0.317387,0.562383,77.191689,238174.0,0.0
2,Hybrid,0.350675,0.500122,42.616992,274324.0,0.0


# Apply Holm correction

In [28]:
p_values = q1q5_train["p_value"].to_numpy()

# Holm-Bonferroni correction
n = len(p_values)
order = np.argsort(p_values)
sorted_p = p_values[order]

adjusted_sorted = np.maximum.accumulate(
    (n - np.arange(n)) * sorted_p
)
adjusted_sorted = np.minimum(adjusted_sorted, 1.0)

p_holm = np.empty(n)
p_holm[order] = adjusted_sorted

q1q5_train["p_holm"] = p_holm
q1q5_train["significant_after_holm"] = p_holm < 0.05

display(q1q5_train.round(10))

,Representation,Q1_mean_error,Q5_mean_error,Increase_percent,U_statistic,p_value,p_holm,significant_after_holm
0,Morgan,0.366646,0.557186,51.968178,262164.0,0.0,0.0,True
1,RDKit,0.317387,0.562383,77.191689,238174.0,0.0,0.0,True
2,Hybrid,0.350675,0.500122,42.616992,274324.0,0.0,0.0,True


In [29]:
distance_error_results.to_csv(
    "scaffold_training_distance_correlations.csv",
    index=False
)

q1q5_train.to_csv(
    "scaffold_training_distance_q1q5.csv",
    index=False
)

pred.to_csv(
    "scaffold_predictions_with_training_distances.csv",
    index=False
)

print("Three training-distance result files saved.")

Three training-distance result files saved.


In [30]:
comparison = pd.DataFrame({
    "Representation": ["Morgan", "RDKit", "Hybrid"],
    "Full_dataset_rho": [0.093129, 0.140185, 0.099445],
    "Training_only_rho": [
        distance_error_results.loc[
            distance_error_results["Representation"] == "Morgan",
            "Spearman_rho"
        ].iloc[0],
        distance_error_results.loc[
            distance_error_results["Representation"] == "RDKit",
            "Spearman_rho"
        ].iloc[0],
        distance_error_results.loc[
            distance_error_results["Representation"] == "Hybrid",
            "Spearman_rho"
        ].iloc[0]
    ]
})

comparison["Change_in_rho"] = (
    comparison["Training_only_rho"] -
    comparison["Full_dataset_rho"]
)

display(comparison.round(6))

,Representation,Full_dataset_rho,Training_only_rho,Change_in_rho
0,Morgan,0.093129,0.179306,0.086177
1,RDKit,0.140185,0.211808,0.071623
2,Hybrid,0.099445,0.146648,0.047203


# Recalculate Spearman correlations with reliable p-values

In [31]:
from scipy.stats import spearmanr

results_precise = []

for rep in ["Morgan", "RDKit", "Hybrid"]:
    rho, p_value = spearmanr(
        pred[f"{rep}_train_distance"],
        pred[f"{rep}_error"]
    )

    results_precise.append({
        "Representation": rep,
        "Spearman_rho": rho,
        "p_value": p_value
    })

precise_correlations = pd.DataFrame(results_precise)

pd.set_option("display.float_format", "{:.3e}".format)
display(precise_correlations)

,Representation,Spearman_rho,p_value
0,Morgan,1.793e-01,5.110e-32
1,RDKit,2.118e-01,2.817e-44
2,Hybrid,1.466e-01,7.535e-22


# Save the precise correlation results

In [32]:
precise_correlations.to_csv(
    "scaffold_training_distance_correlations_precise.csv",
    index=False
)

print("Precise correlation results saved.")

Precise correlation results saved.


## Training-Only Distance Validation — Scaffold Split

### 1. Purpose

To evaluate whether the relationship between chemical-space distance and prediction error changes when the nearest neighbor of each test molecule is selected exclusively from the training set.

### 2. Methodology and Validation

| Parameter | Verified result |
|-----------|-----------------|
| Total molecules | 20,449 |
| Training molecules | 16,202 |
| Test molecules | 4,247 |
| Unique Bemis–Murcko scaffolds | 10,698 |
| Overlapping scaffolds | 0 |
| Missing training-only distances | 0 for all three representations |
| Model retraining | Not performed; existing predictions reused |

Scaffolds were extracted from canonical_smiles using the same method as Notebook 08. The scaffold split was reproduced using GroupShuffleSplit with test_size=0.20 and random_state=42.

Distance definitions:

- Morgan: Tanimoto distance, calculated as 1 - Tanimoto similarity.
- RDKit: Euclidean distance using scaled RDKit descriptors.
- Hybrid: Euclidean distance using the hybrid molecular representation.

For each of the 4,247 test molecules, the nearest neighbor was selected exclusively from the 16,202 training molecules.

### 3. Training-Only Distance–Error Correlations

Spearman correlation was calculated between each test molecule's training-only nearest-neighbor distance and its corresponding absolute prediction error.

| Representation | Spearman rho | p-value |
|----------------|--------------|---------|
| Morgan | 0.179306 | 5.110 × 10⁻³² |
| RDKit | 0.211808 | 2.817 × 10⁻⁴⁴ |
| Hybrid | 0.146648 | 7.535 × 10⁻²² |

Interpretation:

- All three representations showed positive, statistically significant associations between distance to the training set and prediction error.
- RDKit showed the strongest association, followed by Morgan and Hybrid.
- The correlations were modest in magnitude; distance alone does not explain prediction error.

### 4. Q1-versus-Q5 Prediction Error Comparison

Test molecules were divided into five distance quintiles, from Q1 (shortest distances) to Q5 (largest distances). Mean absolute prediction errors were compared between Q1 and Q5 using the Mann–Whitney U test.

| Representation | Q1 mean error | Q5 mean error | Increase in mean error |
|----------------|---------------|---------------|------------------------|
| Morgan | 0.366646 | 0.557186 | 51.97% |
| RDKit | 0.317387 | 0.562383 | 77.19% |
| Hybrid | 0.350675 | 0.500122 | 42.62% |

All three comparisons were marked significant after Holm correction in the notebook output. Exact Holm-adjusted p-values remain to be verified before manuscript reporting.

Interpretation:

- Mean prediction error was higher in Q5 than in Q1 for all three representations.
- RDKit showed the largest increase in mean error (77.19%).
- Hybrid had the lowest Q5 mean error (0.500122).

### 5. Comparison with the Earlier Full-Dataset-Distance Analysis

| Representation | Full-dataset Spearman rho | Training-only Spearman rho | Change in rho |
|----------------|---------------------------|----------------------------|---------------|
| Morgan | 0.093129 | 0.179306 | +0.086177 |
| RDKit | 0.140185 | 0.211808 | +0.071623 |
| Hybrid | 0.099445 | 0.146648 | +0.047203 |

The earlier analysis used nearest-neighbor distances calculated across the complete dataset. The training-only analysis calculates each test molecule's distance to the training set.

The positive distance–error association was observed under both definitions. RDKit had the strongest association in both analyses, although the relative ranking of Morgan and Hybrid changed.

Training-only distances are more directly relevant to evaluating how distance from available training data relates to generalization on unseen scaffolds. The two analyses answer related but distinct questions.

### 6. Interpretation and Limitations

1. Distance to the training set was positively associated with prediction error for all three representations.
2. RDKit showed the strongest distance–error association and the largest Q1-to-Q5 increase in mean error.
3. The associations were modest; distance alone does not explain prediction error.
4. Statistical significance does not establish causality or demonstrate that one representation is universally superior.
5. The original full-dataset-distance analysis is retained for transparency, and the training-only analysis serves as complementary validation.

### 7. Files Saved

| File | Contents |
|------|----------|
| scaffold_training_distance_correlations.csv | Initial training-only Spearman results |
| scaffold_training_distance_correlations_precise.csv | Training-only correlations with precisely displayed p-values |
| scaffold_training_distance_q1q5.csv | Q1-versus-Q5 comparisons and Holm-correction results |
| scaffold_predictions_with_training_distances.csv | Scaffold-test predictions, errors, and all three training-only distances |

### 8. Validation Status

Training-only distances, correlation analyses, and Q1-versus-Q5 comparisons have been completed without retraining the models.

Exact Holm-adjusted p-values still require verification before final manuscript reporting.

In [33]:
import os

files_to_check = [
    "scaffold_training_distance_correlations.csv",
    "scaffold_training_distance_correlations_precise.csv",
    "scaffold_training_distance_q1q5.csv",
    "scaffold_predictions_with_training_distances.csv"
]

for file in files_to_check:
    print(f"{file}: {os.path.exists(file)}")

scaffold_training_distance_correlations.csv: True
scaffold_training_distance_correlations_precise.csv: True
scaffold_training_distance_q1q5.csv: True
scaffold_predictions_with_training_distances.csv: True


# inspect the saved statistical results

In [34]:
import pandas as pd

# Load the saved validation results
q1q5 = pd.read_csv("scaffold_training_distance_q1q5.csv")
corr = pd.read_csv("scaffold_training_distance_correlations_precise.csv")
pred = pd.read_csv("scaffold_predictions_with_training_distances.csv")

# Display all columns and precise numerical values
pd.set_option("display.max_columns", None)
pd.set_option("display.precision", 12)

print("=== Q1 vs Q5 RESULTS ===")
print("Columns:", q1q5.columns.tolist())
display(q1q5)

print("\n=== TRAINING-ONLY CORRELATIONS ===")
print("Columns:", corr.columns.tolist())
display(corr)

print("\n=== SAVED PREDICTIONS AND DISTANCES ===")
print("Shape:", pred.shape)
print("Columns:", pred.columns.tolist())
print("Missing values:", pred.isna().sum().sum())

=== Q1 vs Q5 RESULTS ===
Columns: ['Representation', 'Q1_mean_error', 'Q5_mean_error', 'Increase_percent', 'U_statistic', 'p_value', 'p_holm', 'significant_after_holm']


,Representation,Q1_mean_error,Q5_mean_error,Increase_percent,U_statistic,p_value,p_holm,significant_after_holm
0,Morgan,3.666e-01,5.572e-01,5.197e+01,2.622e+05,8.848e-23,1.770e-22,True
1,RDKit,3.174e-01,5.624e-01,7.719e+01,2.382e+05,4.977e-34,1.493e-33,True
2,Hybrid,3.507e-01,5.001e-01,4.262e+01,2.743e+05,8.737e-18,8.737e-18,True



=== TRAINING-ONLY CORRELATIONS ===
Columns: ['Representation', 'Spearman_rho', 'p_value']


,Representation,Spearman_rho,p_value
0,Morgan,1.793e-01,5.110e-32
1,RDKit,2.118e-01,2.817e-44
2,Hybrid,1.466e-01,7.535e-22



=== SAVED PREDICTIONS AND DISTANCES ===
Shape: (4247, 13)
Columns: ['canonical_smiles', 'scaffold', 'log10_median_papp', 'y_true', 'Morgan_pred', 'Morgan_error', 'RDKit_pred', 'RDKit_error', 'Hybrid_pred', 'Hybrid_error', 'Morgan_train_distance', 'RDKit_train_distance', 'Hybrid_train_distance']
Missing values: 30


In [35]:
print("Missing values by column:")
display(pred.isna().sum())

print("\nRows containing missing values:")
display(pred[pred.isna().any(axis=1)].head(10))

Missing values by column:


canonical_smiles          0
scaffold                 30
log10_median_papp         0
y_true                    0
Morgan_pred               0
Morgan_error              0
RDKit_pred                0
RDKit_error               0
Hybrid_pred               0
Hybrid_error              0
Morgan_train_distance     0
RDKit_train_distance      0
Hybrid_train_distance     0
dtype: int64


Rows containing missing values:


,canonical_smiles,scaffold,log10_median_papp,y_true,Morgan_pred,Morgan_error,RDKit_pred,RDKit_error,Hybrid_pred,Hybrid_error,Morgan_train_distance,RDKit_train_distance,Hybrid_train_distance
115,CC(=O)/C(C)=N/O,NaN,7.782e-01,7.782e-01,1.100e+00,3.218e-01,9.130e-01,1.349e-01,1.019e+00,2.408e-01,8.214e-01,1.078e+01,1.369e+00
116,CC(=O)C(=CN(C)C)C(C)=O,NaN,7.918e-02,7.918e-02,1.048e+00,9.684e-01,9.979e-01,9.187e-01,1.052e+00,9.729e-01,7.879e-01,1.013e+01,1.350e+00
171,CC(=O)N[C@@H](CS)C(=O)O,NaN,-7.058e-02,-7.058e-02,7.137e-01,7.843e-01,-5.112e-02,1.946e-02,3.078e-01,3.784e-01,6.977e-01,8.338e+00,1.075e+00
329,CC(C)(S)C(N)C(=O)O,NaN,1.090e+00,1.090e+00,1.000e+00,9.001e-02,8.564e-02,1.005e+00,4.236e-01,6.667e-01,7.442e-01,1.056e+01,1.236e+00
623,CCC(C)(C)C(=O)O,NaN,1.301e+00,1.301e+00,1.016e+00,2.851e-01,5.801e-01,7.210e-01,1.148e+00,1.527e-01,6.250e-01,7.348e+00,1.122e+00
648,CCCCCCCC/C=C\CCCCCCCC(=O)NC(CO)(CO)CO,NaN,7.046e-01,7.046e-01,1.071e+00,3.662e-01,6.945e-01,1.012e-02,7.880e-01,8.346e-02,7.115e-01,1.604e+01,1.300e+00
650,CCCCCCCCCCCCCCCCCCOCC(COP(=O)([O-])OCC[N+](C)(...,NaN,5.953e-01,5.953e-01,1.634e+00,1.039e+00,9.895e-01,3.942e-01,7.508e-01,1.555e-01,7.705e-01,7.168e+01,1.492e+00
651,CCCCCCCCCCCCCCCCOP(=O)([O-])OCC[N+](C)(C)C,NaN,3.373e-01,3.373e-01,1.680e+00,1.342e+00,1.086e+00,7.485e-01,9.355e-01,5.983e-01,7.255e-01,7.011e+01,1.459e+00
652,CCCCCCCCCCCCOC(=O)CN(C)C(=N)N,NaN,7.918e-02,7.918e-02,1.423e+00,1.343e+00,1.051e+00,9.717e-01,1.044e+00,9.650e-01,6.250e-01,1.327e+01,1.214e+00
657,CCCCCCCCCOC(=O)CN(C)C(=N)N,NaN,1.255e-01,1.255e-01,1.423e+00,1.297e+00,1.181e+00,1.056e+00,1.113e+00,9.879e-01,6.250e-01,1.134e+01,1.235e+00


In [36]:
# Identify exactly which columns contain missing values
print("Missing values by column:")
display(pred.isna().sum().to_frame("Missing_count").query("Missing_count > 0"))

# Check whether prediction errors and training-only distances are complete
critical_cols = [
    "y_true",
    "Morgan_error",
    "RDKit_error",
    "Hybrid_error",
    "Morgan_train_distance",
    "RDKit_train_distance",
    "Hybrid_train_distance"
]

print("\nMissing values in analysis columns:")
display(pred[critical_cols].isna().sum())

# Count missing scaffold values
print("\nMissing scaffold values:", pred["scaffold"].isna().sum())

Missing values by column:


,Missing_count
scaffold,30



Missing values in analysis columns:


y_true                   0
Morgan_error             0
RDKit_error              0
Hybrid_error             0
Morgan_train_distance    0
RDKit_train_distance     0
Hybrid_train_distance    0
dtype: int64


Missing scaffold values: 30


In [37]:
# Inspect molecules with missing scaffold labels
missing_scaffolds = pred[pred["scaffold"].isna()]

print("Number of molecules without scaffold labels:",
      len(missing_scaffolds))

display(
    missing_scaffolds[
        ["canonical_smiles", "log10_median_papp"]
    ].head(10)
)

Number of molecules without scaffold labels: 30


,canonical_smiles,log10_median_papp
115,CC(=O)/C(C)=N/O,7.782e-01
116,CC(=O)C(=CN(C)C)C(C)=O,7.918e-02
171,CC(=O)N[C@@H](CS)C(=O)O,-7.058e-02
329,CC(C)(S)C(N)C(=O)O,1.090e+00
623,CCC(C)(C)C(=O)O,1.301e+00
648,CCCCCCCC/C=C\CCCCCCCC(=O)NC(CO)(CO)CO,7.046e-01
650,CCCCCCCCCCCCCCCCCCOCC(COP(=O)([O-])OCC[N+](C)(...,5.953e-01
651,CCCCCCCCCCCCCCCCOP(=O)([O-])OCC[N+](C)(C)C,3.373e-01
652,CCCCCCCCCCCCOC(=O)CN(C)C(=N)N,7.918e-02
657,CCCCCCCCCOC(=O)CN(C)C(=N)N,1.255e-01


In [38]:
from rdkit import Chem
from rdkit.Chem.Scaffolds import MurckoScaffold

def check_scaffold(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return "Invalid SMILES"

    scaffold = MurckoScaffold.GetScaffoldForMol(mol)

    if scaffold is None or scaffold.GetNumAtoms() == 0:
        return "Empty scaffold (likely acyclic)"

    return "Non-empty scaffold"

missing_scaffolds["scaffold_check"] = (
    missing_scaffolds["canonical_smiles"].apply(check_scaffold)
)

display(missing_scaffolds["scaffold_check"].value_counts())

C:\Users\kiran\AppData\Local\Temp\ipykernel_23328\3801378801.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  missing_scaffolds["scaffold_check"] = (


scaffold_check
Empty scaffold (likely acyclic)    30
Name: count, dtype: int64

In [39]:
import pandas as pd
import numpy as np
import os

# Load the four saved training-only validation files
corr = pd.read_csv("scaffold_training_distance_correlations_precise.csv")
q1q5 = pd.read_csv("scaffold_training_distance_q1q5.csv")
pred = pd.read_csv("scaffold_predictions_with_training_distances.csv")
corr_initial = pd.read_csv("scaffold_training_distance_correlations.csv")

# 1. Check file availability and row counts
print("=== FILE AND ROW-COUNT CHECK ===")
for file in [
    "scaffold_training_distance_correlations.csv",
    "scaffold_training_distance_correlations_precise.csv",
    "scaffold_training_distance_q1q5.csv",
    "scaffold_predictions_with_training_distances.csv"
]:
    print(f"{file}: {os.path.exists(file)}")

print("\nPrediction rows:", len(pred))
print("Unique prediction SMILES:", pred["canonical_smiles"].nunique())
print("Correlation rows:", len(corr))
print("Q1/Q5 rows:", len(q1q5))

# 2. Check all three representations are present
expected = {"Morgan", "RDKit", "Hybrid"}

print("\n=== REPRESENTATION CHECK ===")
print("Correlation representations:", set(corr["Representation"]))
print("Q1/Q5 representations:", set(q1q5["Representation"]))

# 3. Check missing values in analysis columns
analysis_cols = [
    "y_true",
    "Morgan_error", "RDKit_error", "Hybrid_error",
    "Morgan_train_distance",
    "RDKit_train_distance",
    "Hybrid_train_distance"
]

print("\n=== MISSING-VALUE CHECK ===")
display(pred[analysis_cols].isna().sum().to_frame("Missing values"))

# 4. Check that initial and precise correlations agree
print("\n=== CORRELATION FILE COMPARISON ===")
print("Initial correlations:")
display(corr_initial)
print("Precise correlations:")
display(corr)

# 5. Check Q1/Q5 results are complete
print("\n=== Q1/Q5 STATISTICAL CHECK ===")
display(q1q5)

print("\nConsistency checks completed.")

=== FILE AND ROW-COUNT CHECK ===
scaffold_training_distance_correlations.csv: True
scaffold_training_distance_correlations_precise.csv: True
scaffold_training_distance_q1q5.csv: True
scaffold_predictions_with_training_distances.csv: True

Prediction rows: 4247
Unique prediction SMILES: 4247
Correlation rows: 3
Q1/Q5 rows: 3

=== REPRESENTATION CHECK ===
Correlation representations: {'RDKit', 'Morgan', 'Hybrid'}
Q1/Q5 representations: {'RDKit', 'Morgan', 'Hybrid'}

=== MISSING-VALUE CHECK ===


,Missing values
y_true,0
Morgan_error,0
RDKit_error,0
Hybrid_error,0
Morgan_train_distance,0
RDKit_train_distance,0
Hybrid_train_distance,0



=== CORRELATION FILE COMPARISON ===
Initial correlations:


,Representation,Spearman_rho,p_value
0,Morgan,1.793e-01,5.110e-32
1,RDKit,2.118e-01,2.817e-44
2,Hybrid,1.466e-01,7.535e-22


Precise correlations:


,Representation,Spearman_rho,p_value
0,Morgan,1.793e-01,5.110e-32
1,RDKit,2.118e-01,2.817e-44
2,Hybrid,1.466e-01,7.535e-22



=== Q1/Q5 STATISTICAL CHECK ===


,Representation,Q1_mean_error,Q5_mean_error,Increase_percent,U_statistic,p_value,p_holm,significant_after_holm
0,Morgan,3.666e-01,5.572e-01,5.197e+01,2.622e+05,8.848e-23,1.770e-22,True
1,RDKit,3.174e-01,5.624e-01,7.719e+01,2.382e+05,4.977e-34,1.493e-33,True
2,Hybrid,3.507e-01,5.001e-01,4.262e+01,2.743e+05,8.737e-18,8.737e-18,True



Consistency checks completed.


# Independently verify Q1/Q5 calculations

In [40]:
import numpy as np
import pandas as pd
from scipy.stats import mannwhitneyu

pred = pd.read_csv("scaffold_predictions_with_training_distances.csv")

results = []

for rep in ["Morgan", "RDKit", "Hybrid"]:
    distance_col = f"{rep}_train_distance"
    error_col = f"{rep}_error"

    # Assign five distance quintiles
    quintile = pd.qcut(
        pred[distance_col],
        q=5,
        labels=False,
        duplicates="drop"
    ) + 1

    q1 = pred.loc[quintile == 1, error_col].dropna()
    q5 = pred.loc[quintile == 5, error_col].dropna()

    u_stat, p_value = mannwhitneyu(
        q1, q5, alternative="two-sided"
    )

    results.append({
        "Representation": rep,
        "Q1_n": len(q1),
        "Q5_n": len(q5),
        "Q1_mean_error": q1.mean(),
        "Q5_mean_error": q5.mean(),
        "Increase_percent": (
            (q5.mean() - q1.mean()) / q1.mean() * 100
        ),
        "U_statistic": u_stat,
        "p_value": p_value
    })

recalculated_q1q5 = pd.DataFrame(results)
display(recalculated_q1q5.round(10))

,Representation,Q1_n,Q5_n,Q1_mean_error,Q5_mean_error,Increase_percent,U_statistic,p_value
0,Morgan,852,849,3.666e-01,5.572e-01,5.197e+01,2.622e+05,0.000e+00
1,RDKit,850,850,3.174e-01,5.624e-01,7.719e+01,2.382e+05,0.000e+00
2,Hybrid,850,850,3.507e-01,5.001e-01,4.262e+01,2.743e+05,0.000e+00


In [41]:
for name in [
    "train_idx_scaffold",
    "test_idx_scaffold",
    "train_scaffolds",
    "test_scaffolds"
]:
    print(f"{name}: {name in globals()}")

train_idx_scaffold: True
test_idx_scaffold: True
train_scaffolds: True
test_scaffolds: True


In [42]:
import numpy as np

# 1. Verify the original split
print("=== SCAFFOLD SPLIT ===")
print("Training indices:", len(train_idx_scaffold))
print("Test indices:", len(test_idx_scaffold))
print("Overlapping indices:",
      len(set(train_idx_scaffold) & set(test_idx_scaffold)))

print("Overlapping scaffolds:",
      len(train_scaffolds & test_scaffolds))

# 2. Verify the saved distance results cover every test molecule
print("\n=== SAVED DISTANCE COVERAGE ===")
print("Expected test molecules:", len(test_idx_scaffold))
print("Saved prediction molecules:", len(pred))
print("Unique saved SMILES:", pred["canonical_smiles"].nunique())

distance_cols = [
    "Morgan_train_distance",
    "RDKit_train_distance",
    "Hybrid_train_distance"
]

print("\nMissing training-only distances:")
display(pred[distance_cols].isna().sum())

# 3. Check distance values are valid
print("\nNegative distance counts:")
display((pred[distance_cols] < 0).sum())

print("\nInfinite distance counts:")
display(
    pd.DataFrame(
        np.isinf(pred[distance_cols].to_numpy()),
        columns=distance_cols
    ).sum()
)

=== SCAFFOLD SPLIT ===
Training indices: 16202
Test indices: 4247
Overlapping indices: 0
Overlapping scaffolds: 0

=== SAVED DISTANCE COVERAGE ===
Expected test molecules: 4247
Saved prediction molecules: 4247
Unique saved SMILES: 4247

Missing training-only distances:


Morgan_train_distance    0
RDKit_train_distance     0
Hybrid_train_distance    0
dtype: int64


Negative distance counts:


Morgan_train_distance    0
RDKit_train_distance     0
Hybrid_train_distance    0
dtype: int64


Infinite distance counts:


Morgan_train_distance    0
RDKit_train_distance     0
Hybrid_train_distance    0
dtype: int64

In [43]:
expected_test_smiles = (
    df.iloc[test_idx_scaffold]["canonical_smiles"]
    .reset_index(drop=True)
)

saved_test_smiles = pred["canonical_smiles"].reset_index(drop=True)

print(
    "Test molecule order matches:",
    expected_test_smiles.equals(saved_test_smiles)
)

Test molecule order matches: True
